In [ ]:
import ee
import datetime

# Authenticate and initialize Google Earth Engine
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

# USA bounding box
bbox = ee.Geometry.BBox(-171.791110603, 18.91619, -66.96466, 71.3577635769)

# CHIRPS dataset
chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').filterBounds(bbox)

# Range of years
years = list(range(1990, 2025))

print(f"Exporting CHIRPS Daily data for USA: {years[0]}-{years[-1]}")
print(f"Bounding box: {bbox.getInfo()['coordinates']}")
print(f"{'='*60}\n")

# Loop through years
for year in years:
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    # Filter to that year's daily images
    year_coll = chirps.filterDate(start, end).select('precipitation')
    
    # Check if collection has images
    count = year_coll.size().getInfo()
    print(f"Processing {year}: Found {count} daily images")
    
    if count == 0:
        print(f"  ⚠️  No data for {year}, skipping\n")
        continue

    # Merge all days into a single multi-band image
    year_img = year_coll.toBands()

    # Add time metadata
    year_img = year_img.set('system:time_start', start.millis())

    # Export to Drive
    task = ee.batch.Export.image.toDrive(
        image=year_img.clip(bbox),
        description=f"CHIRPS_Daily_USA_{year}",
        folder="USA_CHIRPS_Daily_Years",
        fileNamePrefix=f"CHIRPS_Daily_USA_{year}",
        region=bbox,
        scale=5000,
        crs='EPSG:4326',
        maxPixels=1e13,
        fileFormat='GeoTIFF'
    )
    task.start()
    print(f"  ✓ Export task started: CHIRPS_Daily_USA_{year}")
    print(f"    Task ID: {task.id}\n")

print(f"{'='*60}")
print(f"All {len(years)} export tasks submitted!")
print(f"Monitor tasks at: https://code.earthengine.google.com/tasks")
print(f"{'='*60}")